<a href="https://colab.research.google.com/github/hmabril-sketch/hig-reproducibility/blob/main/notebooks/HIGM48_editor_20x20_VISUAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pintar y analizar HIG₄ y HIG₈ en 20 × 20

1. Ejecuta las tres celdas en orden en Google Colab.
2. Elige uno de los seis colores y pinta con clic o arrastrando el ratón. El color 0 llena inicialmente la imagen.
3. Pulsa **🔬 HIG analysis**: aparecerá a la derecha una copia de la pintura. Debajo, elige HIG₄/HIG₈ y una clase $V_0,V_1,V_2$; pulsa el icono de un nodo para ver sus corners, aristas o píxeles resaltados sobre la copia.
4. Si quieres comparar dos pinturas, analiza la primera y pulsa **Guardar referencia A**; pinta la segunda y vuelve a analizar.
5. Usa la vista de matrices para explorar cualquier bloque de 15 × 15. En cada grafo elige «Completo» o el entorno de un nodo $V_1$; **Descargar matrices y grafos (ZIP)** guarda las matrices y los dos SVG completos.

**Convención de tamaño.** Las regiones tienen área en píxeles; los $1$-generadores geométricos de ambos HIG tienen longitud en aristas primitivas. Los `cut` de $HIG_4$ muestran aristas de su soporte, que puede solaparse con otros soportes. Los nuevos `cut` de $HIG_8^{\mathrm{comb}}$ se indexan por agujeros $4$-conexos: mostramos el área del agujero asociado, sin afirmar que ese área sea soporte del generador. Los $0$-generadores virtuales y `OUT` carecen de área píxel propia.

La imagen es una **segmentación categórica de seis etiquetas**. Se comprueban las identidades $\chi_{\mathrm{gr}}(HIG_4)=2$, $\chi_{\mathrm{gr}}(HIG_8^{\mathrm{comb}})=2$ y $K_8-K_4=D-M$, contando `OUT` en ambos casos. La comparación A/B describe diferencias estructurales entre dibujos; no mide rendimiento predictivo sobre datos clínicos.

In [1]:
from collections import Counter, defaultdict, deque
import numpy as np
from scipy import sparse

# Implementación autocontenida de la construcción intrínseca HIG₄.
# No necesita archivos auxiliares cuando se abre en Colab.

def region_ids(image):
    """Four-connected equal-label components; OUT receives the final id."""
    h, w = image.shape
    ids = np.full((h, w), -1, dtype=np.int32)
    nr = 0
    for y in range(h):
        for x in range(w):
            if ids[y, x] != -1:
                continue
            label = image[y, x]
            ids[y, x] = nr
            queue = deque([(y, x)])
            while queue:
                yy, xx = queue.popleft()
                for ny, nx in ((yy-1, xx), (yy+1, xx), (yy, xx-1), (yy, xx+1)):
                    if 0 <= ny < h and 0 <= nx < w and ids[ny, nx] == -1 and image[ny, nx] == label:
                        ids[ny, nx] = nr
                        queue.append((ny, nx))
            nr += 1
    return ids, nr


def construct(image):
    ids, out = region_ids(image)
    h, w = image.shape
    edges = []                 # (first grid vertex, second, region pair, on frame)
    at = defaultdict(list)
    def add(a, b, u, v, frame):
        if u == v:
            return
        i = len(edges)
        pair = tuple(sorted((int(u), int(v))))
        edges.append((a, b, pair, frame))
        at[a].append(i)
        at[b].append(i)
    for y in range(h):
        for x in range(w+1):
            add((y,x), (y+1,x), out if x == 0 else ids[y,x-1],
                out if x == w else ids[y,x], x in (0,w))
    for y in range(h+1):
        for x in range(w):
            add((y,x), (y,x+1), out if y == 0 else ids[y-1,x],
                out if y == h else ids[y,x], y in (0,h))
    junctions = sorted(v for v, incident in at.items() if len(incident) >= 3)
    jset = set(junctions)
    jindex = {v:i for i,v in enumerate(junctions)}
    geom = []
    seen = set()
    for start in range(len(edges)):
        if start in seen:
            continue
        queue = [start]
        seen.add(start)
        group = []
        for i in queue:
            group.append(i)
            a,b,pair,_ = edges[i]
            for v in (a,b):
                if v in jset:
                    continue
                for nxt in at[v]:
                    if nxt not in seen and edges[nxt][2] == pair:
                        seen.add(nxt)
                        queue.append(nxt)
        endpoints = []
        for i in group:
            for v in edges[i][:2]:
                if v in jset:
                    endpoints.append(v)
        if len(endpoints) == 0:
            typ = "enc0"
        elif len(endpoints) == 2:
            typ = "enc1" if endpoints[0] == endpoints[1] else "seg"
        else:
            raise ValueError(f"Continuation has {len(endpoints)} endpoint occurrences")
        geom.append({"type":typ,"pair":pair,"endpoints":endpoints,"edges":group})
    cuts = []
    seen = set()
    for start in range(len(edges)):
        if start in seen:
            continue
        queue = [start]
        seen.add(start)
        group = []
        frame = False
        for i in queue:
            group.append(i)
            a,b,_,on_frame = edges[i]
            frame |= on_frame
            for v in (a,b):
                for nxt in at[v]:
                    if nxt not in seen:
                        seen.add(nxt)
                        queue.append(nxt)
        if not frame:
            cuts.append(group)
    # Pixel-side to frontier edge; supports local validation on small images.
    pixel_sides = defaultdict(list)
    for i,(a,b,pair,_) in enumerate(edges):
        if a[0] == b[0]:
            yy=a[0]; xx=a[1]
            for py in (yy-1,yy):
                if 0<=py<h:pixel_sides[(py,xx)].append(i)
        else:
            yy=a[0]; xx=a[1]
            for px in (xx-1,xx):
                if 0<=px<w:pixel_sides[(yy,px)].append(i)
    # Each frame-free frontier component corresponds to exactly one
    # bounded 8-component of the complement of exactly one 4-region.
    # Establish the carrier by matching its complete R-boundary to the cut.
    # Its uppermost horizontal edge has the carrier pixel immediately above.
    # This gives a linear-time carrier extraction without flooding the
    # complement separately for every region (which is prohibitive for WSI).
    carriers=[]
    for group in cuts:
        horizontal=[i for i in group if edges[i][0][0]==edges[i][1][0]]
        i=min(horizontal,key=lambda k:(edges[k][0][0],edges[k][0][1]))
        yy,xx=edges[i][0]
        carriers.append(int(ids[yy-1,xx]))
    holes=Counter(carriers)
    if h*w<=10000:
      cut_sets = [set(g) for g in cuts]
      carrier_matches = [[] for _ in cuts]
      verified_holes=Counter()
      for region in range(out):
        visited = np.zeros((h,w), dtype=bool)
        for y in range(h):
            for x in range(w):
                if ids[y,x] == region or visited[y,x]:
                    continue
                queue = deque([(y,x)])
                visited[y,x] = True
                pixels = set()
                frame = False
                while queue:
                    yy,xx = queue.popleft()
                    pixels.add((yy,xx))
                    frame |= yy in (0,h-1) or xx in (0,w-1)
                    for dy in (-1,0,1):
                        for dx in (-1,0,1):
                            ny,nx = yy+dy,xx+dx
                            if 0 <= ny < h and 0 <= nx < w and not visited[ny,nx] and ids[ny,nx] != region:
                                visited[ny,nx] = True
                                queue.append((ny,nx))
                if frame:
                    continue
                verified_holes[region] += 1
                boundary = set()
                for py,px in pixels:
                    for i in pixel_sides[(py,px)]:
                        if region in edges[i][2]:boundary.add(i)
                # C can include frontiers between regions *inside* the hole;
                # only the R--H boundary must lie in that component.
                matches = [k for k,g in enumerate(cut_sets) if boundary and boundary <= g]
                if len(matches) != 1:
                    raise ValueError(f"Cut-hole correspondence failed at region {region}: {len(matches)} matches")
                carrier_matches[matches[0]].append(region)
      if any(m != [carriers[k]] for k,m in enumerate(carrier_matches)) or holes != verified_holes:
          raise ValueError("Fast carrier disagrees with explicit 8-hole verification")
    n0 = len(junctions) + sum(g["type"] == "enc0" for g in geom)
    n1 = len(geom) + len(cuts)
    n2 = out + 1
    ii01,jj01,vv01,ii12,jj12,vv12 = [],[],[],[],[],[]
    virtual = len(junctions)
    for k,g in enumerate(geom):
        endpoints = g["endpoints"]
        if g["type"] == "enc0":
            ii01.append(virtual); jj01.append(k); vv01.append(2)
            virtual += 1
        else:
            for j,count in Counter(jindex[v] for v in endpoints).items():
                ii01.append(j); jj01.append(k); vv01.append(count)
        for region in g["pair"]:
            ii12.append(k); jj12.append(region); vv12.append(1)
    for k,region in enumerate(carriers, start=len(geom)):
        ii12.append(k); jj12.append(region); vv12.append(2)
        # Intentionally NO 0--1 incidence for cut k.
    m01 = sparse.csr_matrix((vv01,(ii01,jj01)),shape=(n0,n1),dtype=np.int32)
    m12 = sparse.csr_matrix((vv12,(ii12,jj12)),shape=(n1,n2),dtype=np.int32)
    return ids, edges, geom, cuts, carriers, holes, m01, m12



In [2]:

import html
import base64
import csv
import io
import json
import zipfile
from statistics import median
from scipy import ndimage

_MATRIX_CACHE = {}
_GRAPH_CACHE = {}
_LAST_ARCHIVE_BYTES = None

def _graph_svg(version, mode='focus', focus_v1=0):
    if version not in _GRAPH_CACHE:
        raise ValueError('Analiza primero la imagen para construir sus grafos.')
    if mode not in ('focus', 'complete'):
        raise ValueError('Modo de grafo desconocido.')
    graph = _GRAPH_CACHE[version]
    m01, m12, nodes = graph['m01'], graph['m12'], graph['nodes']
    n0, n1, n2 = len(nodes['V0']), len(nodes['V1']), len(nodes['V2'])
    assert m01.shape == (n0, n1) and m12.shape == (n1, n2)
    cut_columns = [i for i, node in enumerate(nodes['V1']) if node['type'].startswith('cut')]
    assert m01[:, cut_columns].nnz == 0   # No hay aristas V0--V1cut.
    focus = max(0, min(int(focus_v1), n1-1))

    if mode == 'complete':
        keep1 = set(range(n1))
    else:
        adjacent0 = set(m01[:, focus].nonzero()[0].tolist())
        adjacent2 = set(m12[focus, :].nonzero()[1].tolist())
        # Se expande por V0 y regiones imagen; OUT se representa pero no
        # arrastra a todos los contactos del marco a la vista local.
        other2 = sorted(adjacent2 - {n2-1})
        candidates = {focus}
        if adjacent0:
            candidates.update(m01[sorted(adjacent0), :].nonzero()[1].tolist())
        if other2:
            candidates.update(m12[:, other2].nonzero()[0].tolist())
        keep1 = set([focus] + [i for i in sorted(candidates) if i != focus][:79])

    m01part = m01[:, sorted(keep1)].tocoo()
    chosen1 = sorted(keep1)
    edges01 = [(int(row), chosen1[int(col)], int(value))
               for row, col, value in zip(m01part.row, m01part.col, m01part.data)]
    m12part = m12[chosen1, :].tocoo()
    edges12 = [(chosen1[int(row)], int(col), int(value))
               for row, col, value in zip(m12part.row, m12part.col, m12part.data)]
    keep0 = sorted({row for row, _, _ in edges01})
    keep2 = sorted({col for _, col, _ in edges12})
    if mode == 'complete':
        keep0, keep2 = list(range(n0)), list(range(n2))

    largest = max(len(keep0), len(chosen1), len(keep2), 1)
    spacing = 29 if largest <= 150 else max(6, min(22, 8100 // largest))
    radius = 6 if spacing >= 20 else 3
    height = max(400, 83 + (largest+1)*spacing)
    x = {'V0': 95, 'V1': 535, 'V2': 975}
    y0 = {v: 76 + (i+1)*spacing for i, v in enumerate(keep0)}
    y1 = {v: 76 + (i+1)*spacing for i, v in enumerate(chosen1)}
    y2 = {v: 76 + (i+1)*spacing for i, v in enumerate(keep2)}
    label_all = largest <= 90

    def node_color(kind, node):
        typ = node['type']
        if typ.startswith('cut'):
            return '#ed7544'
        if typ == 'diagonal_typed':
            return '#8c55bd'
        if typ == 'OUT':
            return '#5a6471'
        if 'enclosing' in typ or typ.startswith('enc'):
            return '#2b9b78'
        return {'V0': '#2980ae', 'V1': '#386fc0', 'V2': '#6676d7'}[kind]

    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1070 {height}" '
             f'width="1070" height="{height}" role="img" '
             f'aria-label="Grafo bipartito {html.escape(version)}">',
             '<rect width="1070" height="100%" fill="white"/>',
             '<g fill="#233548" font-size="18" font-family="sans-serif" font-weight="bold">'
             '<text x="62" y="39">V₀</text><text x="505" y="39">V₁</text>'
             '<text x="945" y="39">V₂</text></g>']
    show_multiplicity = (len(edges01) + len(edges12)) <= 85
    for row, col, value in edges01:
        y_a, y_b = y0[row], y1[col]
        parts.append(f'<line x1="{x["V0"]}" y1="{y_a}" x2="{x["V1"]}" y2="{y_b}" '
                     f'stroke="#77a9c3" stroke-opacity=".62" stroke-width="{1 if value==1 else 3}">'
                     f'<title>HIGM_0,1[{row},{col}] = {value}</title></line>')
        if value > 1 and show_multiplicity:
            parts.append(f'<text x="{(x["V0"]+x["V1"])/2}" y="{(y_a+y_b)/2-3}" '
                         f'fill="#173c5c" font-size="12">×{value}</text>')
    for row, col, value in edges12:
        y_a, y_b = y1[row], y2[col]
        parts.append(f'<line x1="{x["V1"]}" y1="{y_a}" x2="{x["V2"]}" y2="{y_b}" '
                     f'stroke="#a79cce" stroke-opacity=".65" stroke-width="{1 if value==1 else 3}">'
                     f'<title>HIGM_1,2[{row},{col}] = {value}</title></line>')
        if value > 1 and show_multiplicity:
            parts.append(f'<text x="{(x["V1"]+x["V2"])/2}" y="{(y_a+y_b)/2-3}" '
                         f'fill="#56357d" font-size="12">×{value}</text>')
    for kind, values, positions in [('V0', keep0, y0), ('V1', chosen1, y1), ('V2', keep2, y2)]:
        for idx in values:
            node = nodes[kind][idx]
            fill = node_color(kind, node)
            tip = html.escape(json.dumps(node, ensure_ascii=False), quote=True)
            parts.append(f'<circle cx="{x[kind]}" cy="{positions[idx]}" r="{radius}" '
                         f'fill="{fill}" stroke="#233548" stroke-width=".6"><title>{tip}</title></circle>')
            if label_all or node['type'] == 'OUT' or (kind == 'V1' and node['type'].startswith('cut')):
                shift = 12 if kind != 'V0' else -12
                anchor = 'start' if kind != 'V0' else 'end'
                parts.append(f'<text x="{x[kind]+shift}" y="{positions[idx]+4}" '
                             f'font-family="sans-serif" font-size="11" fill="#223240" '
                             f'text-anchor="{anchor}">{kind}:{idx}</text>')
    parts.append('</svg>')
    description = (f'{version}: {len(keep0)}/{n0} nodos V₀; '
                   f'{len(chosen1)}/{n1} nodos V₁; {len(keep2)}/{n2} nodos V₂; '
                   f'{len(edges01)} incidencias 0–1 y {len(edges12)} incidencias 1–2 visibles.')
    return {'svg': ''.join(parts), 'description': description, 'focus': focus}

def _graph_window(version, mode, focus_v1):
    result = _graph_svg(str(version), str(mode), focus_v1)
    return {'html': ('<p>' + html.escape(result['description']) +
            ' Los nodos naranjas son cuts: se conectan solo con V₂. '
            'Las líneas gruesas y ×2 indican multiplicidad 2. '
            'Pasa el cursor sobre una arista o nodo para ver sus índices.</p>' +
            '<div class="hig-graph-scroll">' + result['svg'] + '</div>')}

def _matrix_summary(name, matrix):
    return {'name': name, 'rows': int(matrix.shape[0]), 'cols': int(matrix.shape[1]),
            'nnz': int(matrix.nnz), 'sum': int(matrix.sum()),
            'max': int(matrix.data.max()) if matrix.nnz else 0}

def _matrix_window(name, row_start=0, col_start=0):
    if name not in _MATRIX_CACHE:
        raise ValueError('Analiza primero la imagen para generar sus matrices.')
    matrix = _MATRIX_CACHE[name]
    height, width = matrix.shape
    row_start = max(0, min(int(row_start), max(0, height-1)))
    col_start = max(0, min(int(col_start), max(0, width-1)))
    rr, cc = min(row_start+15, height), min(col_start+15, width)
    block = matrix[row_start:rr, col_start:cc].toarray()
    header = '<th>fila / columna</th>' + ''.join(f'<th>{j}</th>' for j in range(col_start, cc))
    body = []
    for index, row in enumerate(block, start=row_start):
        cells = ''.join(f'<td class="hig-matrix-value hig-matrix-{min(2,int(v))}">{int(v)}</td>'
                        for v in row)
        body.append(f'<tr><th>{index}</th>{cells}</tr>')
    label = 'filas V₀ · columnas V₁' if name.endswith('M01') else 'filas V₁ · columnas V₂'
    result = (f'<p><strong>{html.escape(name)}</strong> ({height} × {width}); {label}. '
              'Ventana de hasta 15 × 15; 0 = ausencia, 1 = incidencia simple, 2 = multiplicidad doble.</p>'
              '<div class="hig-table-wrap"><table class="hig-matrix"><thead><tr>'
              + header + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>')
    return {'html': result}

def _matrix_archive(image, matrices, metadata, graph_svgs):
    global _LAST_ARCHIVE_BYTES
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        labels = io.BytesIO()
        np.save(labels, image, allow_pickle=False)
        zf.writestr('labels_20x20.npy', labels.getvalue())
        zf.writestr('node_metadata.json', json.dumps(metadata, ensure_ascii=False, indent=2))
        for name, svg in graph_svgs.items():
            zf.writestr(name + '_graph.svg', svg)
        for name, matrix in matrices.items():
            binary = io.BytesIO()
            sparse.save_npz(binary, matrix.tocsr())
            zf.writestr(f'{name}.npz', binary.getvalue())
            triples = io.StringIO()
            writer = csv.writer(triples)
            writer.writerow(['row', 'col', 'multiplicity'])
            coo = matrix.tocoo()
            writer.writerows(sorted(zip(coo.row.tolist(), coo.col.tolist(), coo.data.tolist())))
            zf.writestr(f'{name}_triplets.csv', triples.getvalue())
        zf.writestr('README.txt',
            'Incidencias enteras NO orientadas de HIG4 y HIG8 combinatorio.\n'
            'HIGM_0_1: filas V0, columnas V1. HIGM_1_2: filas V1, columnas V2.\n'
            'Índices base 0; OUT es el último nodo de V2.\n'
            'Los CSV contienen todos los valores NO NULOS y su multiplicidad.\n'
            'Los NPZ contienen la matriz dispersa completa, incluidos ceros implícitos.\n'
            'Usa scipy.sparse.load_npz("HIG4_M01.npz") y consulta node_metadata.json.\n'
            'Estas incidencias no son matrices de borde orientadas.\n')
    _LAST_ARCHIVE_BYTES = bundle.getvalue()
    return base64.b64encode(_LAST_ARCHIVE_BYTES).decode('ascii')

def descargar_matrices():
    if _LAST_ARCHIVE_BYTES is None:
        raise RuntimeError('Primero pulsa HIG analysis en el editor.')
    from google.colab import files
    path = '/content/HIG4_HIG8_matrices_20x20.zip'
    with open(path, 'wb') as destination:
        destination.write(_LAST_ARCHIVE_BYTES)
    files.download(path)

def _esc(value):
    return html.escape(str(value), quote=True)

def _table(headers, rows):
    head = ''.join(f'<th>{_esc(x)}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{_esc(x)}</td>' for x in row) + '</tr>' for row in rows)
    return f'<div class="hig-table-wrap"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

def _distribution(values):
    if not values:
        return '—'
    return f'{min(values)} / {median(values):g} / {max(values)}'

def _graph_descriptors(m01, m12):
    n0, n1 = m01.shape
    assert m12.shape[0] == n1
    n2 = m12.shape[1]
    vertex_count = n0 + n1 + n2
    edges01, edges12 = int(m01.sum()), int(m12.sum())
    edge_count = edges01 + edges12  # Las entradas 2 cuentan dos aristas paralelas.
    parent = list(range(vertex_count))

    def root(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(a, b):
        a, b = root(int(a)), root(int(b))
        if a != b:
            parent[b] = a

    for row, col in zip(*m01.nonzero()):
        union(row, n0 + col)
    for row, col in zip(*m12.nonzero()):
        union(n0 + row, n0 + n1 + col)
    components = len({root(node) for node in range(vertex_count)})
    cycle_rank = edge_count - vertex_count + components
    graph_euler = vertex_count - edge_count
    assert cycle_rank >= 0 and graph_euler == components - cycle_rank
    return {'V': vertex_count, 'E01': edges01, 'E12': edges12,
            'E': edge_count, 'components': components,
            'beta1': cycle_rank, 'chi_graph': graph_euler}

def _histogram(values):
    return [[int(value), int(count)] for value, count in sorted(Counter(values).items())]

def _derive_hig8(image, ids, geom, cuts, holes4, m01, m12, junctions):
    h, w = image.shape
    old_out = m12.shape[1] - 1
    parent = list(range(old_out))

    def root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        a, b = root(a), root(b)
        if a != b:
            parent[max(a, b)] = min(a, b)

    # q_v se lee sobre los cuatro cuadrantes completos de cada vértice interior.
    for y in range(1, h):
        for x in range(1, w):
            for (a, b) in (((y-1, x-1), (y, x)),
                           ((y-1, x), (y, x-1))):
                ra, rb = int(ids[a]), int(ids[b])
                if ra != rb and image[a] == image[b]:
                    union(ra, rb)

    roots = sorted({root(r) for r in range(old_out)})
    index = {r: i for i, r in enumerate(roots)}
    quotient = [index[root(r)] for r in range(old_out)]
    new_out = len(roots)
    quotient.append(new_out)    # OUT se transporta a OUT.
    qmap = np.asarray(quotient, dtype=int)[ids]
    junction_index = {v: i for i, v in enumerate(junctions)}
    diagonal = []
    d_by_q = Counter()
    for y in range(1, h):
        for x in range(1, w):
            nw, ne = int(qmap[y-1, x-1]), int(qmap[y-1, x])
            sw, se = int(qmap[y, x-1]), int(qmap[y, x])
            for qid, same, other_a, other_b, orientation in (
                (nw, se, ne, sw, 'NW–SE'),
                (ne, sw, nw, se, 'NE–SW')):
                if qid == same and other_a != qid and other_b != qid:
                    assert (y, x) in junction_index
                    diagonal.append({'vertex': (y, x), 'region': qid,
                                     'junction': junction_index[(y, x)],
                                     'orientation': orientation})
                    d_by_q[qid] += 1

    # Oráculo independiente para la agrupación: componentes 8 directas.
    direct = []
    for value in np.unique(image):
        labels, count = ndimage.label(image == value, structure=np.ones((3, 3), dtype=int))
        direct.extend((labels == i).tobytes() for i in range(1, count + 1))
    assert sorted(direct) == sorted((qmap == i).tobytes() for i in range(new_out))

    four_structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=int)
    hole_nodes = []
    regions = []
    for qid in range(new_out):
        mask = qmap == qid
        complement = np.pad(~mask, 1, constant_values=True)
        labels, count = ndimage.label(complement, structure=four_structure)
        outside = int(labels[0, 0])
        holes = [labels[1:-1, 1:-1] == i for i in range(1, count + 1) if i != outside]
        members = [r for r in range(old_out) if quotient[r] == qid]
        k = len(members)
        d = d_by_q[qid]
        h4 = sum(holes4.get(r, 0) for r in members)
        assert len(holes) == h4 + d - k + 1, (qid, k, d, h4, len(holes))
        coords = np.argwhere(mask)
        y, x = coords[0]
        regions.append({'id': qid, 'label': int(image[y, x]), 'pixels': len(coords),
                        'members': members, 'holes': len(holes), 'd': d,
                        'cycle_rank': d-k+1})
        for hole in holes:
            hole_coords = np.argwhere(hole)
            hole_nodes.append({'carrier': qid, 'hole_pixels': len(hole_coords),
                               'first_pixel': tuple(int(c) for c in hole_coords[0]),
                               'pixels': hole_coords.tolist()})

    m = old_out - new_out
    d = len(diagonal)
    k4, k8 = len(cuts), len(hole_nodes)
    n0 = m01.shape[0] + d
    n1 = len(geom) + k8
    n2 = new_out + 1
    chi = n0 - n1 + n2
    assert k8-k4 == d-m
    assert chi == 2

    # Incidencias tipadas de la definición de HIG_8^{comb}.
    old_to_new = sparse.csr_matrix((np.ones(old_out+1, dtype=int),
                    (np.arange(old_out+1), quotient)), shape=(old_out+1, n2))
    retained12 = m12[:len(geom), :] @ old_to_new
    cut12 = sparse.csr_matrix((np.full(k8, 2, dtype=int),
                    (np.arange(k8), [item['carrier'] for item in hole_nodes])),
                    shape=(k8, n2))
    m12_8 = sparse.vstack([retained12, cut12], format='csr')
    old01 = m01[:, :len(geom)]
    new_rows = old01[[item['junction'] for item in diagonal], :]
    geometric01 = sparse.vstack([old01, new_rows], format='csr')
    m01_8 = sparse.hstack([geometric01, sparse.csr_matrix((n0, k8), dtype=int)], format='csr')
    assert m01_8.shape == (n0, n1) and m12_8.shape == (n1, n2)
    assert m01_8[:, len(geom):].nnz == 0
    assert all(m12_8[len(geom)+i, item['carrier']] == 2
               for i, item in enumerate(hole_nodes))

    return {'regions': regions, 'diagonal': diagonal, 'holes': hole_nodes,
            'quotient': quotient, 'm': m, 'd': d, 'k4': k4, 'k8': k8,
            'n0': n0, 'n1': n1, 'n2': n2, 'chi': chi,
            'm01_total': int(m01_8.sum()), 'm12_total': int(m12_8.sum()),
            'm01': m01_8, 'm12': m12_8}

def _analyse_hig4(labels):
    if not isinstance(labels, list) or len(labels) != 400:
        raise ValueError('Se necesitan exactamente 400 etiquetas.')
    if any(type(x) is not int or not 0 <= x <= 5 for x in labels):
        raise ValueError('Cada píxel debe tener una etiqueta entera entre 0 y 5.')
    image = np.asarray(labels, dtype=np.int32).reshape(20, 20)
    ids, edges, geom, cuts, carriers, holes, m01, m12 = construct(image)
    n0, n1 = m01.shape
    n2 = m12.shape[1]
    out = n2 - 1
    chi = n0 - n1 + n2
    counts = Counter(g['type'] for g in geom)
    degree = Counter(v for a, b, _, _ in edges for v in (a, b))
    junctions = sorted(v for v, d in degree.items() if d >= 3)
    virtual = [(i, g) for i, g in enumerate(geom) if g['type'] == 'enc0']
    assert n0 == len(junctions) + len(virtual)
    assert n1 == len(geom) + len(cuts)
    assert chi == 2
    assert int(m01[:, len(geom):].nnz) == 0
    assert sum(holes.values()) == len(cuts)
    graph4 = _graph_descriptors(m01, m12)

    groups = [
        ('V₀ junction', len(junctions), [1] * len(junctions)),
        ('V₀ enclosing virtual', len(virtual), []),
        ('V₁ segmentation', counts['seg'], [len(g['edges']) for g in geom if g['type'] == 'seg']),
        ('V₁ enclosing, loop', counts['enc1'], [len(g['edges']) for g in geom if g['type'] == 'enc1']),
        ('V₁ enclosing, closed cycle', counts['enc0'], [len(g['edges']) for g in geom if g['type'] == 'enc0']),
        ('V₁ cut', len(cuts), [len(c) for c in cuts]),
        ('V₂ image region', out, np.bincount(ids.ravel(), minlength=out).tolist()),
        ('V₂ OUT', 1, []),
    ]
    summary_rows = [(name, cardinal, _distribution(sizes)) for name, cardinal, sizes in groups]

    zero_rows = []
    for i, v in enumerate(junctions):
        zero_rows.append((f'v₀:{i}', 'junction', 1, f'({v[0]}, {v[1]})', degree[v]))
    for j, (geom_id, g) in enumerate(virtual, start=len(junctions)):
        zero_rows.append((f'v₀:{j}', 'enclosing virtual', '—', f'ciclo v₁:{geom_id}', len(g['edges'])))

    one_rows = []
    for i, g in enumerate(geom):
        one_rows.append((f'v₁:{i}', g['type'], len(g['edges']),
                         str(g['pair']), str(g['endpoints'])))
    for j, (component, carrier) in enumerate(zip(cuts, carriers), start=len(geom)):
        one_rows.append((f'v₁:{j}', 'cut', len(component),
                         f'portador v₂:{carrier}', 'sin incidencia 0–1'))

    two_rows = []
    for i in range(out):
        loc = np.argwhere(ids == i)
        y, x = loc[0]
        two_rows.append((f'v₂:{i}', int(image[y, x]), len(loc), f'({y}, {x})'))
    two_rows.append((f'v₂:{out}', 'OUT', '—', 'exterior compactificado'))

    hig8 = _derive_hig8(image, ids, geom, cuts, holes, m01, m12, junctions)
    graph8 = _graph_descriptors(hig8['m01'], hig8['m12'])
    qregions, occurrences, new_holes = hig8['regions'], hig8['diagonal'], hig8['holes']
    eight_groups = [
        ('V₀ junction (retenidos)', len(junctions), [1] * len(junctions)),
        ('V₀ enclosing virtual (retenidos)', len(virtual), []),
        ('V₀ diagonal tipado z', len(occurrences), [1] * len(occurrences)),
        ('V₁ segmentation (retenidos)', counts['seg'], [len(g['edges']) for g in geom if g['type'] == 'seg']),
        ('V₁ enclosing loop (retenidos)', counts['enc1'], [len(g['edges']) for g in geom if g['type'] == 'enc1']),
        ('V₁ enclosing ciclo (retenidos)', counts['enc0'], [len(g['edges']) for g in geom if g['type'] == 'enc0']),
        ('V₁ cut: área del agujero asociado', len(new_holes), [item['hole_pixels'] for item in new_holes]),
        ('V₂ región 8-conexa', len(qregions), [r['pixels'] for r in qregions]),
        ('V₂ OUT', 1, []),
    ]
    eight_summary = [(name, cardinal, _distribution(sizes)) for name, cardinal, sizes in eight_groups]
    diag_rows = [(f'z:{n0+i}', f'({z["vertex"][0]}, {z["vertex"][1]})',
                  f'Q:{z["region"]}', z['orientation'], f'junction v₀:{z["junction"]}')
                 for i, z in enumerate(occurrences)]
    hole_rows = [(f'c:{len(geom)+i}', f'Q:{item["carrier"]}', item['hole_pixels'],
                  str(item['first_pixel']), 'incidencia 1–2 = 2; columna 0–1 = 0')
                 for i, item in enumerate(new_holes)]
    region_rows = [(f'Q:{r["id"]}', r['label'], r['pixels'], str(r['members']),
                    r['holes'], r['d'], r['cycle_rank']) for r in qregions]

    nodes0_4 = ([{'index': i, 'type': 'junction', 'vertex': list(v)}
                 for i, v in enumerate(junctions)] +
                [{'index': len(junctions)+i, 'type': 'enclosing_virtual',
                  'associated_V1': geom_id}
                 for i, (geom_id, _) in enumerate(virtual)])
    nodes1_geom = [{'index': i, 'type': g['type'], 'regions4': list(g['pair']),
                    'support_frontier_edges': len(g['edges'])}
                   for i, g in enumerate(geom)]
    nodes1_4 = nodes1_geom + [
        {'index': len(geom)+i, 'type': 'cut4', 'carrier4': int(carrier),
         'support_frontier_edges': len(component)}
        for i, (component, carrier) in enumerate(zip(cuts, carriers))]
    nodes2_4 = []
    for i in range(out):
        y, x = np.argwhere(ids == i)[0]
        nodes2_4.append({'index': i, 'type': 'region4', 'label': int(image[y, x]),
                         'pixels': int(np.count_nonzero(ids == i))})
    nodes2_4.append({'index': out, 'type': 'OUT'})
    nodes0_8 = nodes0_4 + [
        {'index': n0+i, 'type': 'diagonal_typed', 'vertex': list(z['vertex']),
         'region8': int(z['region']), 'duplicate_junction_row': int(z['junction'])}
        for i, z in enumerate(occurrences)]
    nodes1_8 = [{**g, 'regions8': [int(hig8['quotient'][r]) for r in g['regions4']]}
                for g in nodes1_geom] + [
        {'index': len(geom)+i, 'type': 'cut8_4hole', 'carrier8': int(item['carrier']),
         'associated_hole_pixels': int(item['hole_pixels'])}
        for i, item in enumerate(new_holes)]
    nodes2_8 = [{'index': int(r['id']), 'type': 'region8', 'label': int(r['label']),
                 'pixels': int(r['pixels']), 'constituent_regions4': r['members']}
                for r in qregions] + [{'index': len(qregions), 'type': 'OUT'}]
    matrix_nodes = {'HIG4': {'V0': nodes0_4, 'V1': nodes1_4, 'V2': nodes2_4},
                    'HIG8': {'V0': nodes0_8, 'V1': nodes1_8, 'V2': nodes2_8}}
    # El visor conserva los mismos índices que HIGM_01 y HIGM_12.
    # Los testigos de nodos abstractos se marcan explícitamente como asociados.
    def edge_support(indices):
        return [[list(edges[int(k)][0]), list(edges[int(k)][1])] for k in indices]

    geom_support = [edge_support(g['edges']) for g in geom]
    common_zero = ([{'index': i, 'type': 'junction', 'kind': 'point',
                     'point': list(v), 'size': 1} for i, v in enumerate(junctions)] +
                   [{'index': len(junctions)+i, 'type': 'enclosing virtual',
                     'kind': 'associated_edges', 'edges': geom_support[geom_id],
                     'size': len(g['edges']), 'associated_v1': geom_id}
                    for i, (geom_id, g) in enumerate(virtual)])
    common_geom_one = [{'index': i, 'type': g['type'], 'kind': 'edges',
                        'edges': geom_support[i], 'size': len(g['edges'])}
                       for i, g in enumerate(geom)]
    region4_pixels = [np.argwhere(ids == i).tolist() for i in range(out)]
    region8_pixels = [np.argwhere(np.isin(ids, r['members'])).tolist() for r in qregions]
    node_geometry = {
        'HIG4': {
            'V0': common_zero,
            'V1': common_geom_one + [
                {'index': len(geom)+i, 'type': 'cut4', 'kind': 'edges',
                 'edges': edge_support(component), 'size': len(component),
                 'carrier': int(carrier)}
                for i, (component, carrier) in enumerate(zip(cuts, carriers))],
            'V2': [{'index': i, 'type': 'region4', 'kind': 'pixels',
                    'pixels': points, 'size': len(points), 'label': nodes2_4[i]['label']}
                   for i, points in enumerate(region4_pixels)] +
                  [{'index': out, 'type': 'OUT', 'kind': 'exterior', 'size': None}],
        },
        'HIG8': {
            'V0': common_zero + [
                {'index': n0+i, 'type': 'diagonal typed', 'kind': 'point',
                 'point': list(z['vertex']), 'size': 1,
                 'carrier': int(z['region']), 'orientation': z['orientation']}
                for i, z in enumerate(occurrences)],
            'V1': common_geom_one + [
                {'index': len(geom)+i, 'type': 'cut8',
                 'kind': 'associated_pixels', 'pixels': item['pixels'],
                 'size': item['hole_pixels'], 'carrier': int(item['carrier'])}
                for i, item in enumerate(new_holes)],
            'V2': [{'index': int(r['id']), 'type': 'region8', 'kind': 'pixels',
                    'pixels': region8_pixels[i], 'size': int(r['pixels']),
                    'label': int(r['label'])} for i, r in enumerate(qregions)] +
                  [{'index': len(qregions), 'type': 'OUT',
                    'kind': 'exterior', 'size': None}],
        },
    }
    for version in ('HIG4', 'HIG8'):
        for layer in ('V0', 'V1', 'V2'):
            assert len(node_geometry[version][layer]) == len(matrix_nodes[version][layer])
            assert all(i == item['index'] for i, item in enumerate(node_geometry[version][layer]))
    matrices = {'HIG4_M01': m01.tocsr(), 'HIG4_M12': m12.tocsr(),
                'HIG8_M01': hig8['m01'], 'HIG8_M12': hig8['m12']}
    for version in ('HIG4', 'HIG8'):
        dims = matrix_nodes[version]
        assert matrices[f'{version}_M01'].shape == (len(dims['V0']), len(dims['V1']))
        assert matrices[f'{version}_M12'].shape == (len(dims['V1']), len(dims['V2']))
    _MATRIX_CACHE.clear()
    _MATRIX_CACHE.update(matrices)
    _GRAPH_CACHE.clear()
    _GRAPH_CACHE.update({
        'HIG4': {'m01': m01.tocsr(), 'm12': m12.tocsr(), 'nodes': matrix_nodes['HIG4']},
        'HIG8': {'m01': hig8['m01'], 'm12': hig8['m12'], 'nodes': matrix_nodes['HIG8']},
    })
    matrix_info = [_matrix_summary(name, matrix) for name, matrix in matrices.items()]
    graph_svgs = {name: _graph_svg(name, 'complete', 0)['svg'] for name in ('HIG4', 'HIG8')}
    archive_b64 = _matrix_archive(image, matrices, {
        'index_base': 0, 'image_shape': [20, 20], 'node_sets': matrix_nodes,
        'graph_descriptors': {'HIG4': graph4, 'HIG8': graph8},
        'matrices': {item['name']: {'shape': [item['rows'], item['cols']],
                                    'nnz': item['nnz'], 'sum': item['sum']}
                     for item in matrix_info}}, graph_svgs)
    graph_info = []
    for name in ('HIG4', 'HIG8'):
        node_sets = matrix_nodes[name]
        cut_first = next((item['index'] for item in node_sets['V1']
                          if item['type'].startswith('cut')), 0)
        total = sum(len(node_sets[layer]) for layer in ('V0', 'V1', 'V2'))
        graph_info.append({'name': name,
                           'default_mode': 'complete' if total <= 150 else 'focus',
                           'focus': cut_first, 'total_nodes': total})

    markup = f'''
    <div class="hig-analysis">
      <div class="hig-cards">
        <div><strong>|V₀|</strong><span>{n0}</span></div>
        <div><strong>|V₁|</strong><span>{n1}</span></div>
        <div><strong>|V₂| (incluye OUT)</strong><span>{n2}</span></div>
        <div><strong>χ = |V₀| − |V₁| + |V₂|</strong><span>{n0} − {n1} + {n2} = {chi}</span></div>
      </div>
      <h3>Grafo bipartito HIG₄</h3>
      <p>V = {n0} + {n1} + {n2} = <strong>{graph4['V']}</strong>;
      E = ΣHIGM₀,₁ + ΣHIGM₁,₂ = {graph4['E01']} + {graph4['E12']}
      = <strong>{graph4['E']}</strong> (con multiplicidad).</p>
      <p>χ<sub>grafo</sub> = V − E = <strong>{graph4['chi_graph']}</strong>;
      componentes c = {graph4['components']}; β₁ = {graph4['beta1']}.
      Control: χ<sub>grafo</sub> = c − β₁. El recuento graduado mostrado arriba vale {chi}.</p>
      <h3>Cardinales y tamaños del soporte</h3>
      <p>En cada fila: mínimo / mediana / máximo. «—» indica ausencia de tamaño geométrico propio.</p>
      {_table(('Tipo', 'Cardinal', 'Tamaño mín. / med. / máx.'), summary_rows)}
      <details><summary>V₀ · {n0} nodos (ver detalles)</summary>
        {_table(('Nodo', 'Tipo', 'Tamaño', 'Ubicación o ciclo asociado', 'Grado o longitud'), zero_rows)}
      </details>
      <details><summary>V₁ · {n1} nodos (ver detalles)</summary>
        {_table(('Nodo', 'Tipo', 'Aristas del soporte', 'Regiones o portador', 'Extremos'), one_rows)}
      </details>
      <details><summary>V₂ · {n2} nodos (ver detalles)</summary>
        {_table(('Nodo', 'Etiqueta', 'Píxeles', 'Píxel inicial / descripción'), two_rows)}
      </details>
      <p class="hig-foot">Verificado: χ = 2; {len(cuts)} cuts = {sum(holes.values())} agujeros
      (4-región / 8-complemento); columnas 0–1 de los cuts nulas.</p>
    </div>'''
    markup += f'''
    <div class="hig-analysis">
      <h2>HIG₈ combinatorio derivado</h2>
      <div class="hig-cards">
        <div><strong>|V₀⁽⁸⁾|</strong><span>{hig8['n0']}</span></div>
        <div><strong>|V₁⁽⁸⁾|</strong><span>{hig8['n1']}</span></div>
        <div><strong>|V₂⁽⁸⁾| (incluye OUT)</strong><span>{hig8['n2']}</span></div>
        <div><strong>χ₈</strong><span>{hig8['n0']} − {hig8['n1']} + {hig8['n2']} = {hig8['chi']}</span></div>
      </div>
      <h3>Grafo bipartito HIG₈</h3>
      <p>V = {hig8['n0']} + {hig8['n1']} + {hig8['n2']}
      = <strong>{graph8['V']}</strong>;
      E = ΣHIGM₀,₁ + ΣHIGM₁,₂ = {graph8['E01']} + {graph8['E12']}
      = <strong>{graph8['E']}</strong> (con multiplicidad).</p>
      <p>χ<sub>grafo</sub> = V − E = <strong>{graph8['chi_graph']}</strong>;
      componentes c = {graph8['components']}; β₁ = {graph8['beta1']}.
      Control: χ<sub>grafo</sub> = c − β₁. El recuento graduado mostrado arriba vale {hig8['chi']}.</p>
      <p>Quotient: M = {hig8['m']} regiones fusionadas; D = {hig8['d']}
      ocurrencias diagonales; K₄ = {hig8['k4']}; K₈ = {hig8['k8']}.
      Verificado: K₈ − K₄ = {hig8['k8']-hig8['k4']} = D − M.</p>
      <p>Los (1)-generadores geométricos conservan su tamaño en aristas;
      para los nuevos cuts se indica el área del agujero complementario que los indexa.</p>
      {_table(('Tipo', 'Cardinal', 'Tamaño mín. / med. / máx.'), eight_summary)}
      <details><summary>V₀: {len(occurrences)} nuevos nodos diagonales tipados</summary>
        {_table(('Nodo', 'Vértice', 'Región 8', 'Diagonal', 'Fila 0–1 duplicada de'), diag_rows)}
      </details>
      <details><summary>V₁: {len(new_holes)} nuevos cuts por agujeros 4-conexos</summary>
        {_table(('Nodo', 'Portador', 'Píxeles del agujero', 'Primer píxel', 'Incidencia'), hole_rows)}
      </details>
      <details><summary>V₂: {len(qregions)} regiones 8-conexas y OUT</summary>
        {_table(('Nodo', 'Etiqueta', 'Píxeles', 'Regiones 4 constituyentes',
                 'Agujeros 4', 'd', 'd − k + 1'), region_rows)}
        <p>OUT es el generador adicional, sin área píxel.</p>
      </details>
      <p class="hig-foot">Incidencias reconstruidas: Σμ₀,₁ = {hig8['m01_total']},
      Σμ₁,₂ = {hig8['m12_total']}. Las filas diagonales duplican las de su junction
      y las columnas de los nuevos cuts en μ₀,₁ son nulas.</p>
    </div>'''
    matrix_rows = [(item['name'], f"{item['rows']} × {item['cols']}",
                    item['nnz'], item['sum'], item['max']) for item in matrix_info]
    matrix_options = ''.join(f'<option value="{item["name"]}">{item["name"]}</option>'
                             for item in matrix_info)
    markup += f'''
    <div class="hig-analysis">
      <h2>Matrices HIGM₀,₁ y HIGM₁,₂</h2>
      <p>Incidencias enteras sin orientación. HIGM₀,₁: filas V₀ y columnas V₁;
      HIGM₁,₂: filas V₁ y columnas V₂. Los valores no nulos pueden ser 1 o 2.
      Los índices empiezan en 0; OUT es la última columna V₂.</p>
      {_table(('Matriz', 'Dimensiones', 'Entradas no nulas', 'Suma de incidencias', 'Máximo'), matrix_rows)}
      <div class="toolbar">
        <label>Matriz <select data-matrix-name>{matrix_options}</select></label>
        <label>Fila inicial <input data-matrix-row type="number" min="0" value="0" step="1"></label>
        <label>Columna inicial <input data-matrix-col type="number" min="0" value="0" step="1"></label>
        <button data-matrix-show>Ver ventana 15 × 15</button>
      </div>
      <div data-matrix-window></div>
      <p class="hig-foot">El ZIP descargable contiene las cuatro matrices completas
      como NPZ dispersos y CSV de todas las entradas no nulas, además de los
      índices y tipos de nodos de cada fila y columna, y los dos grafos SVG completos.</p>
    </div>'''
    graph_sections = []
    for info in graph_info:
        complete = ' selected' if info['default_mode'] == 'complete' else ''
        focus = ' selected' if info['default_mode'] == 'focus' else ''
        graph_sections.append(f'''
        <section class="hig-graph-panel" data-graph-version="{info['name']}">
          <h3>Grafo {info['name']} · V₁ con V₀ ∪ V₂</h3>
          <p>{info['total_nodes']} nodos tipados. En imágenes densas se inicia
          con un entorno local; «Completo» dibuja todos los nodos y aristas.</p>
          <div class="toolbar">
            <label>Vista <select data-graph-mode>
              <option value="focus"{focus}>Entorno de un V₁</option>
              <option value="complete"{complete}>Grafo completo</option>
            </select></label>
            <label>Índice V₁ <input data-graph-focus type="number" min="0"
              value="{info['focus']}" step="1"></label>
            <button data-graph-render>Dibujar grafo</button>
          </div>
          <div data-graph-window></div>
        </section>''')
    markup += '<div class="hig-analysis"><h2>Grafos de incidencia bipartitos</h2>' + \
              ''.join(graph_sections) + '</div>'
    area4 = np.bincount(ids.ravel(), minlength=out).tolist()
    features = {
        'HIG₄ · vértices V del grafo': graph4['V'],
        'HIG₄ · aristas E del grafo': graph4['E'],
        'HIG₄ · Euler del grafo V−E': graph4['chi_graph'],
        'HIG₄ · componentes del grafo': graph4['components'],
        'HIG₄ · ciclos β₁ del grafo': graph4['beta1'],
        'HIG₄ · junctions': len(junctions),
        'HIG₄ · enclosing 0': len(virtual),
        'HIG₄ · segmentos 1': counts['seg'],
        'HIG₄ · enclosing 1 tipo 0 / tipo 1': [counts['enc0'], counts['enc1']],
        'HIG₄ · cuts': len(cuts),
        'HIG₄ · regiones 4': out,
        'HIG₄ · distribución de áreas regionales': _histogram(area4),
        'HIG₄ · distribución de longitudes de contacto': _histogram([len(g['edges']) for g in geom]),
        'HIG₈ · vértices V del grafo': graph8['V'],
        'HIG₈ · aristas E del grafo': graph8['E'],
        'HIG₈ · Euler del grafo V−E': graph8['chi_graph'],
        'HIG₈ · componentes del grafo': graph8['components'],
        'HIG₈ · ciclos β₁ del grafo': graph8['beta1'],
        'HIG₈ · regiones 8': len(qregions),
        'HIG₈ · fusiones M': hig8['m'],
        'HIG₈ · ocurrencias diagonales D': hig8['d'],
        'HIG₈ · cuts por agujeros 4': hig8['k8'],
        'HIG₈ · distribución de áreas regionales': _histogram([r['pixels'] for r in qregions]),
        'HIG₈ · agujeros por región': _histogram([r['holes'] for r in qregions]),
        'HIG₈ · distribución de áreas de agujeros': _histogram([item['hole_pixels'] for item in new_holes]),
        'HIG₈ · ciclos de contacto diagonal': sum(r['cycle_rank'] for r in qregions),
    }
    return {'html': markup, 'features': features, 'matrices': matrix_info,
            'node_geometry': node_geometry,
            'graph_descriptors': {'HIG4': graph4, 'HIG8': graph8},
            'graphs': graph_info,
            'archive_b64': archive_b64,
            'counts': {'hig4': {'v0': n0, 'v1': n1, 'v2': n2, 'chi': chi,
                                'V': graph4['V'], 'E': graph4['E'],
                                'chi_graph': graph4['chi_graph'], 'cuts': len(cuts)},
                       'hig8': {'v0': hig8['n0'], 'v1': hig8['n1'], 'v2': hig8['n2'],
                                'chi': hig8['chi'], 'V': graph8['V'], 'E': graph8['E'],
                                'chi_graph': graph8['chi_graph'], 'cuts': hig8['k8'],
                                'diagonal': hig8['d'], 'merged': hig8['m']}}}

# El botón del lienzo invoca esta función en el kernel de Colab.
from IPython.display import JSON
from google.colab import output
def _colab_hig_callback(labels):
    return JSON(_analyse_hig4(labels))
def _colab_matrix_callback(name, row_start, col_start):
    return JSON(_matrix_window(name, row_start, col_start))
def _colab_graph_callback(version, mode, focus_v1):
    return JSON(_graph_window(version, mode, focus_v1))
output.register_callback('hig20.analyze', _colab_hig_callback)
output.register_callback('hig20.matrix_window', _colab_matrix_callback)
output.register_callback('hig20.graph_window', _colab_graph_callback)
print('Motores HIG₄ y HIG₈ combinatorio listos: tamaños, incidencias e identidades.')


Motores HIG₄ y HIG₈ combinatorio listos: tamaños, incidencias e identidades.


In [3]:

from IPython.display import HTML, display
from uuid import uuid4
import json

widget_id = 'hig20-' + uuid4().hex
style = """
<style>
.hig20 {font-family:system-ui,sans-serif;max-width:1120px;color:#17212b}
.hig20 .toolbar{display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin:10px 0}
.hig20 .hig-images{display:grid;grid-template-columns:repeat(2,minmax(250px,1fr));gap:16px;align-items:start}
.hig20 .hig-images h3{font-size:15px;margin:5px 0}.hig20 .hig-images canvas{display:block}
@media(max-width:650px){.hig20 .hig-images{grid-template-columns:1fr}}
.hig20 .hig-node-inspector{border:1px solid #cad6df;border-radius:8px;padding:12px;margin:12px 0;background:#f8fafb}
.hig20 .hig-node-inspector h3{margin:0 0 6px}.hig20 .hig-node-menu{display:flex;gap:7px;flex-wrap:wrap;margin:8px 0}
.hig20 .hig-node-menu button.active,.hig20 .hig-node-list button.active{background:#173b64;color:white}
.hig20 .hig-node-list{display:flex;flex-wrap:wrap;gap:5px;margin:10px 0;max-height:180px;overflow:auto}
.hig20 .hig-node-list button{padding:5px 8px;font-weight:500;font-size:12px}
.hig20 .hig-node-detail{padding:7px 10px;background:white;border-left:4px solid #d78117;margin:8px 0}
.hig20 .hig-page{display:flex;gap:8px;align-items:center;flex-wrap:wrap}
.hig20 [data-snapshot]{cursor:default}
.hig20 button{border:1px solid #a6b2bd;border-radius:8px;background:white;padding:8px 11px;cursor:pointer;font-weight:600}
.hig20 button:hover{background:#edf3f7}.hig20 button.selected{outline:3px solid #27384e;outline-offset:1px}
.hig20 .analyse{background:#173b64;color:white;border-color:#173b64}
.hig20 .analyse:hover{background:#28557f;color:white}
.hig20 canvas{width:min(100%,520px);height:auto;touch-action:none;cursor:crosshair;border:2px solid #263746;border-radius:6px;image-rendering:pixelated}
.hig20 .hig-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(145px,1fr));gap:8px;margin:12px 0}
.hig20 .hig-cards>div{padding:12px;border:1px solid #cad6df;background:#f4f8fa;border-radius:8px}
.hig20 .hig-cards strong,.hig20 .hig-cards span{display:block}.hig20 .hig-cards span{font-size:20px;margin-top:5px}
.hig20 table{border-collapse:collapse;width:100%;font-size:13px}
.hig20 td,.hig20 th{padding:5px 7px;border-bottom:1px solid #dce4e9;text-align:left;white-space:nowrap}
.hig20 th{background:#f0f4f7}.hig20 .hig-table-wrap{max-height:370px;overflow:auto;margin:7px 0 15px}
.hig20 details{margin:8px 0;border:1px solid #dce4e9;padding:8px;border-radius:8px}
.hig20 summary{cursor:pointer;font-weight:600}.hig20 .hig-foot{font-size:13px;color:#405364}
.hig20 input,.hig20 select{padding:7px;border:1px solid #a6b2bd;border-radius:6px}
.hig20 input[type=number]{width:75px}.hig20 .hig-matrix td,.hig20 .hig-matrix th{text-align:center;min-width:24px}
.hig20 .hig-matrix-0{background:#f3f5f7;color:#6d7981}
.hig20 .hig-matrix-1{background:#aed3ec;color:#102b41;font-weight:600}
.hig20 .hig-matrix-2{background:#f3ad71;color:#552b12;font-weight:700}
.hig20 .hig-graph-panel{border:1px solid #d5e0e8;border-radius:8px;padding:10px;margin:12px 0}
.hig20 .hig-graph-scroll{max-height:640px;max-width:100%;overflow:auto;border:1px solid #cad6df;background:white}
.hig20 .hig-graph-scroll svg{display:block;max-width:none}
</style>
"""
markup = f"""
<div id="{widget_id}" class="hig20">
 <h2>Editor HIG₄ / HIG₈ · 20 × 20</h2>
 <p>Elige una etiqueta y pinta con el ratón o el dedo. Cada casilla es un píxel.</p>
 <div class="toolbar" data-brushes></div>
 <div class="hig-images">
  <div><h3>Imagen editable</h3><canvas data-paint width="520" height="520" aria-label="Lienzo editable de 20 por 20 píxeles"></canvas></div>
  <div><h3>Copia del último análisis</h3><canvas data-snapshot width="520" height="520" aria-label="Copia analizada con el nodo seleccionado resaltado"></canvas></div>
 </div>
 <div data-node-inspector class="hig-node-inspector" aria-live="polite">Pulsa HIG analysis para explorar sus nodos sobre la copia.</div>
 <div class="toolbar">
  <button class="analyse" data-analyse>🔬 HIG analysis</button>
  <button data-download disabled>Descargar matrices y grafos (ZIP)</button>
  <button data-save-a>Guardar referencia A</button>
  <button data-clear>Limpiar (color 0)</button>
  <button data-demo>Ejemplo con encierro</button>
  <span data-status role="status">Pinta la imagen y pulsa HIG analysis.</span>
 </div>
 <div data-compare></div>
 <div data-results></div>
</div>"""

js = r"""
(() => {
 const root = document.getElementById(WIDGET_ID);
 const canvas = root.querySelector('[data-paint]');
 const ctx = canvas.getContext('2d');
 const snapshotCanvas = root.querySelector('[data-snapshot]');
 const snapshotCtx = snapshotCanvas.getContext('2d');
 const inspector = root.querySelector('[data-node-inspector]');
 const colors = ['#3478b7','#f08328','#3da65b','#9a60be','#d64f60','#e6c648'];
 const pixels = Array(400).fill(0);
 const brushes = root.querySelector('[data-brushes]');
 const status = root.querySelector('[data-status]');
 const results = root.querySelector('[data-results]');
 const comparison = root.querySelector('[data-compare]');
 const download=root.querySelector('[data-download]');
 let brush = 0, dragging = false, revision = 0, last = null, baseline = null;
 let snapshotPixels = null, geometry = null, layer = ['HIG4','V0'];
 let selectedNode = null, nodePage = 0;
 colors.forEach((color, index) => {
   const button = document.createElement('button');
   button.textContent = `Color ${index}`;
   button.title = `Etiqueta ${index}`;
   button.style.borderLeft = `12px solid ${color}`;
   button.addEventListener('click', () => {
     brush = index;
     [...brushes.querySelectorAll('button')].forEach((b,i) => b.classList.toggle('selected',i===index));
   });
   brushes.appendChild(button);
 });
 brushes.querySelector('button').classList.add('selected');
 function drawGrid(target, data) {
   for (let y=0;y<20;y++) for (let x=0;x<20;x++) {
     target.fillStyle = colors[data[y*20+x]];
     target.fillRect(x*26,y*26,26,26);
   }
   target.strokeStyle = 'rgba(12,29,48,.32)';target.lineWidth=1;
   for(let i=0;i<=20;i++){
     target.beginPath();target.moveTo(i*26+.5,0);target.lineTo(i*26+.5,520);target.stroke();
     target.beginPath();target.moveTo(0,i*26+.5);target.lineTo(520,i*26+.5);target.stroke();
   }
 }
 function draw(){drawGrid(ctx,pixels)}
 function drawSnapshot(){
   snapshotCtx.clearRect(0,0,520,520);
   if(!snapshotPixels){snapshotCtx.fillStyle='#f3f6f9';snapshotCtx.fillRect(0,0,520,520);
     snapshotCtx.fillStyle='#42576a';snapshotCtx.font='18px system-ui';
     snapshotCtx.fillText('Copia disponible tras HIG analysis',40,260);return;}
   drawGrid(snapshotCtx,snapshotPixels);
   if(!selectedNode)return;
   const node=geometry?.[selectedNode[0]]?.[selectedNode[1]]?.[selectedNode[2]];
   if(!node)return;
   snapshotCtx.save();
   if(node.kind==='pixels'||node.kind==='associated_pixels'){
     for(const [y,x] of node.pixels){
       snapshotCtx.fillStyle='rgba(255,240,0,.52)';snapshotCtx.fillRect(x*26,y*26,26,26);
       snapshotCtx.strokeStyle='#572500';snapshotCtx.lineWidth=2;
       snapshotCtx.strokeRect(x*26+2,y*26+2,22,22);
     }
   }else if(node.kind==='point'){
     const [y,x]=node.point;
     snapshotCtx.beginPath();snapshotCtx.arc(x*26,y*26,9,0,Math.PI*2);
     snapshotCtx.fillStyle='#fff566';snapshotCtx.fill();snapshotCtx.lineWidth=3;
     snapshotCtx.strokeStyle='#9c1725';snapshotCtx.stroke();
   }else if(node.kind==='edges'||node.kind==='associated_edges'){
     snapshotCtx.lineCap='round';snapshotCtx.lineWidth=6;
     snapshotCtx.strokeStyle=node.kind==='associated_edges'?'#9c35df':'#fff566';
     if(node.kind==='associated_edges')snapshotCtx.setLineDash([10,6]);
     for(const [[ay,ax],[by,bx]] of node.edges){
       snapshotCtx.beginPath();snapshotCtx.moveTo(ax*26,ay*26);
       snapshotCtx.lineTo(bx*26,by*26);snapshotCtx.stroke();
     }
   }else if(node.kind==='exterior'){
     snapshotCtx.strokeStyle='#fff566';snapshotCtx.lineWidth=8;
     snapshotCtx.setLineDash([16,9]);snapshotCtx.strokeRect(4,4,512,512);
   }
   snapshotCtx.restore();
 }
 function nodeDescription(node,version,rank){
   let relation='Soporte propio';
   if(node.kind==='associated_edges')relation=`Testigo asociado: ciclo del V₁:${node.associated_v1}; el 0-nodo virtual no ocupa un corner.`;
   if(node.kind==='associated_pixels')relation=`Píxeles del agujero 4-conexo asociado al cut₈, portado por V₂:${node.carrier}; no son soporte intrínseco del 1-nodo.`;
   if(node.kind==='exterior')relation='OUT: el marco discontinuo indica el exterior; no tiene soporte píxel finito.';
   if(node.kind==='point')relation=`Corner (${node.point[0]}, ${node.point[1]})${node.carrier===undefined?'':`; ocurrencia tipada de V₂:${node.carrier}`}.`;
   if(node.kind==='edges')relation=`${node.edges.length} aristas primitivas del soporte${node.carrier===undefined?'':`; portador V₂:${node.carrier}`}.`;
   if(node.kind==='pixels')relation=`${node.pixels.length} píxeles de la región${node.label===undefined?'':`; etiqueta ${node.label}`}.`;
   return `${version} · ${rank}:${node.index} · ${node.type} · ${relation}`;
 }
 function showNode(nodeIndex){
   const [version,rank]=layer, node=geometry[version][rank][nodeIndex];
   if(!node)return;
   selectedNode=[version,rank,nodeIndex];drawSnapshot();renderInspector();
 }
 function renderInspector(){
   inspector.replaceChildren();
   if(!geometry){inspector.textContent='Pulsa HIG analysis para explorar sus nodos sobre la copia.';return;}
   const title=document.createElement('h3');title.textContent='Explorar nodos de HIG₄ y HIG₈';inspector.appendChild(title);
   const menu=document.createElement('div');menu.className='hig-node-menu';
   for(const version of ['HIG4','HIG8'])for(const rank of ['V0','V1','V2']){
     const button=document.createElement('button');
     button.textContent=`${version==='HIG4'?'HIG₄':'HIG₈'} · ${rank.replace('0','₀').replace('1','₁').replace('2','₂')} (${geometry[version][rank].length})`;
     button.classList.toggle('active',version===layer[0]&&rank===layer[1]);
     button.addEventListener('click',()=>{layer=[version,rank];nodePage=0;
       selectedNode=null;drawSnapshot();renderInspector();});menu.appendChild(button);
   }
   inspector.appendChild(menu);
   const nodes=geometry[layer[0]][layer[1]], perPage=48;
   const pages=Math.max(1,Math.ceil(nodes.length/perPage));nodePage=Math.min(nodePage,pages-1);
   const pagebar=document.createElement('div');pagebar.className='hig-page';
   const prev=document.createElement('button');prev.textContent='◀ Anterior';prev.disabled=nodePage===0;
   prev.addEventListener('click',()=>{nodePage--;renderInspector()});pagebar.appendChild(prev);
   const span=document.createElement('span');span.textContent=`Página ${nodePage+1}/${pages}`;pagebar.appendChild(span);
   const next=document.createElement('button');next.textContent='Siguiente ▶';next.disabled=nodePage===pages-1;
   next.addEventListener('click',()=>{nodePage++;renderInspector()});pagebar.appendChild(next);
   const jump=document.createElement('input');jump.type='number';jump.min='0';jump.max=String(nodes.length-1);
   jump.placeholder='Índice del nodo';jump.setAttribute('aria-label','Índice del nodo');pagebar.appendChild(jump);
   const go=document.createElement('button');go.textContent='Ir al nodo';
   go.addEventListener('click',()=>{const i=Number(jump.value);
     if(jump.value!==''&&Number.isInteger(i)&&i>=0&&i<nodes.length){nodePage=Math.floor(i/perPage);showNode(i)}});
   pagebar.appendChild(go);inspector.appendChild(pagebar);
   const list=document.createElement('div');list.className='hig-node-list';
   for(const node of nodes.slice(nodePage*perPage,(nodePage+1)*perPage)){
     const button=document.createElement('button');
     const icon=node.type==='diagonal typed'?'◆':node.kind==='point'?'●':
       node.kind==='associated_edges'?'◌':
       node.kind==='associated_pixels'||node.type==='cut4'?'✂':
       node.type==='enc0'||node.type==='enc1'?'↻':node.kind==='edges'?'━':
       node.kind==='exterior'?'◎':'▣';
     button.textContent=`${icon} ${layer[1]}:${node.index}`;
     button.title=nodeDescription(node,layer[0],layer[1]);
     button.classList.toggle('active',selectedNode?.[0]===layer[0]&&selectedNode?.[1]===layer[1]&&selectedNode?.[2]===node.index);
     button.addEventListener('click',()=>showNode(node.index));list.appendChild(button);
   }
   inspector.appendChild(list);
   const detail=document.createElement('div');detail.className='hig-node-detail';
   if(selectedNode){const [version,rank,i]=selectedNode;
     detail.textContent=nodeDescription(geometry[version][rank][i],version,rank);
   }else detail.textContent='Pulsa un icono para resaltar ese nodo en la copia.';
   inspector.appendChild(detail);
 }
 function resetSnapshot(){snapshotPixels=null;geometry=null;selectedNode=null;drawSnapshot();renderInspector()}
 function paint(event) {
   const box=canvas.getBoundingClientRect();
   const x=Math.floor((event.clientX-box.left)*20/box.width);
   const y=Math.floor((event.clientY-box.top)*20/box.height);
   if(x<0||x>=20||y<0||y>=20)return;
   const i=y*20+x;
   if(pixels[i]!==brush){pixels[i]=brush;revision++;draw();}
   status.textContent=`Píxel (${y}, ${x}): etiqueta ${pixels[i]}`;
 }
 canvas.addEventListener('pointerdown',e=>{dragging=true;canvas.setPointerCapture(e.pointerId);paint(e);});
 canvas.addEventListener('pointermove',e=>{if(dragging)paint(e);});
 canvas.addEventListener('pointerup',()=>{dragging=false;});
 canvas.addEventListener('pointercancel',()=>{dragging=false;});
 root.querySelector('[data-clear]').addEventListener('click',()=>{
   pixels.fill(0);revision++;last=null;download.disabled=true;draw();resetSnapshot();results.innerHTML='';comparison.innerHTML='';status.textContent='Lienzo limpio.';
 });
 root.querySelector('[data-demo]').addEventListener('click',()=>{
   pixels.fill(0);
   for(let y=3;y<=16;y++)for(let x=3;x<=16;x++)pixels[y*20+x]=1;
   for(let y=6;y<=13;y++)for(let x=6;x<=13;x++)pixels[y*20+x]=2;
   for(let y=9;y<=10;y++)for(let x=9;x<=10;x++)pixels[y*20+x]=3;
   revision++;last=null;download.disabled=true;draw();resetSnapshot();results.innerHTML='';comparison.innerHTML='';status.textContent='Ejemplo cargado: pulsa HIG analysis.';
 });
 download.addEventListener('click',()=>{
   if(!last || last.revision!==revision || !last.archive){status.textContent='Analiza la pintura actual antes de descargar.';return;}
   const binary=atob(last.archive),bytes=new Uint8Array(binary.length);
   for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);
   const url=URL.createObjectURL(new Blob([bytes],{type:'application/zip'}));
   const link=document.createElement('a');link.href=url;link.download='HIG4_HIG8_matrices_20x20.zip';
   document.body.appendChild(link);link.click();link.remove();
   setTimeout(()=>URL.revokeObjectURL(url),30000);
   status.textContent='Descarga solicitada: matrices y metadatos de la imagen actual.';
 });
 root.querySelector('[data-save-a]').addEventListener('click',()=>{
   if(!last || last.revision!==revision){status.textContent='Analiza la imagen actual antes de guardarla como A.';return;}
   baseline={pixels:pixels.slice(),features:structuredClone(last.features)};
   comparison.innerHTML='';status.textContent='Referencia A guardada. Pinta B y vuelve a analizar.';
 });
 function showComparison(features) {
   comparison.innerHTML='';
   if(!baseline)return;
   const changed=Object.keys(features).filter(key=>
     JSON.stringify(features[key])!==JSON.stringify(baseline.features[key]));
   const differingPixels=pixels.filter((v,i)=>v!==baseline.pixels[i]).length;
   const title=document.createElement('h3');title.textContent=`Comparación A/B · ${differingPixels} píxeles con etiqueta distinta`;
   comparison.appendChild(title);
   const note=document.createElement('p');
   const newEight=changed.filter(key=>key.startsWith('HIG₈'));
   note.textContent=changed.length
     ? `${changed.length} descriptores agregados cambian; ${newEight.length} pertenecen a HIG₈.`
     : 'Las firmas agregadas coinciden; eso no demuestra que las imágenes sean idénticas.';
   comparison.appendChild(note);
   if(changed.length){
     const container=document.createElement('div');container.className='hig-table-wrap';
     const table=document.createElement('table');
     const head=document.createElement('thead');const header=document.createElement('tr');
     ['Descriptor','A','B'].forEach(text=>{const th=document.createElement('th');th.textContent=text;header.appendChild(th)});
     head.appendChild(header);table.appendChild(head);
     const body=document.createElement('tbody');
     changed.forEach(key=>{
       const row=document.createElement('tr');
       [key,JSON.stringify(baseline.features[key]),JSON.stringify(features[key])].forEach(text=>{
         const cell=document.createElement('td');cell.textContent=text;row.appendChild(cell);
       });body.appendChild(row);
     });
     table.appendChild(body);container.appendChild(table);comparison.appendChild(container);
   }
 }
 const analyse=root.querySelector('[data-analyse]');
 async function showMatrix(){
   if(!last || last.revision!==revision){status.textContent='Vuelve a analizar la imagen antes de consultar sus matrices.';return;}
   const choice=results.querySelector('[data-matrix-name]');
   const row=results.querySelector('[data-matrix-row]');
   const col=results.querySelector('[data-matrix-col]');
   const windowBox=results.querySelector('[data-matrix-window]');
   if(!choice || !windowBox)return;
   windowBox.textContent='Cargando matriz…';
   try {
     const reply=await google.colab.kernel.invokeFunction('hig20.matrix_window',
       [choice.value,Number(row.value||0),Number(col.value||0)],{});
     const raw=reply.data?.['application/json'] ?? reply.data?.['text/plain'];
     const data=typeof raw==='string'?JSON.parse(raw):raw;
     if(!data || !data.html)throw new Error('Respuesta del kernel sin ventana de matriz.');
     windowBox.innerHTML=data.html;
   } catch(err){windowBox.textContent='Error: '+String(err.message||err);}
 }
 async function showGraph(panel){
   if(!last || last.revision!==revision){status.textContent='Vuelve a analizar la imagen antes de dibujar el grafo.';return;}
   const mode=panel.querySelector('[data-graph-mode]').value;
   const focus=Number(panel.querySelector('[data-graph-focus]').value||0);
   const box=panel.querySelector('[data-graph-window]');
   const requestedRevision=revision;
   box.textContent='Dibujando grafo…';
   try {
     const reply=await google.colab.kernel.invokeFunction('hig20.graph_window',
       [panel.dataset.graphVersion,mode,focus],{});
     const raw=reply.data?.['application/json'] ?? reply.data?.['text/plain'];
     const data=typeof raw==='string'?JSON.parse(raw):raw;
     if(!data || !data.html)throw new Error('Respuesta del kernel sin grafo.');
     if(requestedRevision===revision)box.innerHTML=data.html;
   } catch(err){box.textContent='Error: '+String(err.message||err);}
 }
 analyse.addEventListener('click',async()=>{
   analyse.disabled=true;status.textContent='Calculando HIG₄ y HIG₈…';
   const snapshot=revision;
   const requestedPixels=pixels.slice();
   try {
     const reply=await google.colab.kernel.invokeFunction('hig20.analyze',[requestedPixels],{});
     const raw=reply.data?.['application/json'] ?? reply.data?.['text/plain'];
     const data=typeof raw==='string'?JSON.parse(raw):raw;
     if(!data || !data.html)throw new Error('Respuesta del kernel sin resultados. Ejecuta de nuevo la celda del motor.');
     results.innerHTML=data.html;
     last={revision:snapshot,features:data.features,archive:data.archive_b64};
     snapshotPixels=requestedPixels;geometry=data.node_geometry;
     layer=['HIG4','V0'];selectedNode=null;nodePage=0;
     drawSnapshot();renderInspector();
     download.disabled=snapshot!==revision;
     if(snapshot===revision)showComparison(data.features);
     status.textContent=snapshot===revision?'Análisis completado.':'Análisis completado para la imagen anterior: vuelve a pulsar para actualizar.';
     const matrixButton=results.querySelector('[data-matrix-show]');
     if(matrixButton){matrixButton.addEventListener('click',showMatrix);if(snapshot===revision)await showMatrix();}
     for(const panel of results.querySelectorAll('[data-graph-version]')){
       panel.querySelector('[data-graph-render]').addEventListener('click',()=>showGraph(panel));
       if(snapshot===revision)await showGraph(panel);
     }
   } catch(err){status.textContent='Error: '+String(err.message||err);}
   finally{analyse.disabled=false;}
 });
 draw();drawSnapshot();
})();
""".replace('WIDGET_ID', json.dumps(widget_id))
display(HTML(style + markup + '<script>' + js + '</script>'))


## Lectura de los resultados

- **Visor de nodos**: la imagen derecha es una copia fija del último análisis. Los iconos $V_0$, $V_1$ y $V_2$ se paginan y pueden localizarse por índice, que coincide con las filas o columnas de HIGM. Los puntos marcan corners de la retícula, las líneas marcan aristas primitivas y las casillas resaltadas marcan píxeles. El ciclo morado de un $V_0$ virtual es un testigo asociado; los píxeles amarillos de un `cut₈` son el agujero asociado. El recuadro discontinuo de `OUT` señala solo el exterior. Si editas la imagen izquierda, la copia y su selección siguen mostrando el análisis anterior hasta volver a pulsar el botón.
- **$HIG_4$**: $V_0^{\mathrm{junc}}$ ocupa un vértice; $V_0^{\mathrm{enc}}$ es virtual. Los segmentos y ciclos de $V_1^{\mathrm{seg}}\sqcup V_1^{\mathrm{enc}}$ se miden en aristas; el soporte de un $V_1^{\mathrm{cut}}$ también se mide en aristas y puede solaparse con los anteriores. Las $V_2^{\mathrm{img}}$ son regiones $4$-conexas medidas en píxeles.
- **$HIG_8^{\mathrm{comb}}$**: se conservan los generadores geométricos; cada ocurrencia diagonal $(v,\mathcal R)$ crea un $0$-generador tipado $z_{v,\mathcal R}$. Cada agujero $4$-conexo del complemento de una región $8$-conexa crea un nuevo `cut`, cuyo tamaño asociado mostrado es el área del agujero. Las $V_2^{(8)}$ se miden en píxeles.
- **`OUT`**: se incluye en ambos cardinales de $V_2$ y no tiene área finita. Los dos valores de Euler son **recuentos alternados de generadores tipados**; el segundo no afirma una nueva homología ni un complejo celular plano regular.
- **Matrices HIGM**: para cada representación, $HIGM_{0,1}$ tiene filas $V_0$ y columnas $V_1$; $HIGM_{1,2}$ tiene filas $V_1$ y columnas $V_2$. Son matrices de **incidencia entera no orientada**, con entradas 0, 1 o 2; no se interpretan como operadores de borde de un complejo de cadenas.
- **Grafos bipartitos**: $V_0$ aparece a la izquierda, $V_1$ en el centro y $V_2$ a la derecha. Cada entrada no nula de HIGM dibuja una arista con su multiplicidad; los cortes naranjas de $V_1$ solo se conectan a su $2$-nodo portador. Las vistas locales mantienen las incidencias de los nodos visibles y no representan el grafo completo.
- **Descriptores del grafo**: $V=|V_0|+|V_1|+|V_2|$ incluye `OUT`; $E=\sum HIGM_{0,1}+\sum HIGM_{1,2}$ cuenta las incidencias con multiplicidad. El número de Euler **del grafo** es $\chi_{\mathrm{grafo}}=V-E=c-\beta_1$ y, en general, **no es 2**. El valor 2 corresponde al recuento **graduado** $|V_0|-|V_1|+|V_2|$.

El ZIP contiene `HIG4_M01.npz`, `HIG4_M12.npz`, `HIG8_M01.npz`, `HIG8_M12.npz`, los correspondientes `*_triplets.csv`, `HIG4_graph.svg`, `HIG8_graph.svg`, `node_metadata.json` con el orden y tipo de cada generador, y `labels_20x20.npy`. Para cargar una matriz: `from scipy.sparse import load_npz; M = load_npz('HIG8_M12.npz')`. Los ceros están representados implícitamente en los NPZ. Si el navegador impide descargar desde el botón, ejecuta `descargar_matrices()` en una celda nueva después de analizar la imagen.

El botón vuelve a construir las dos representaciones sobre la imagen actual y comprueba las identidades locales y globales. La comparación A/B enumera los descriptores que cambian; firmas agregadas idénticas no garantizan imágenes idénticas. No utiliza un clasificador entrenado.